# Sistemas Jerárquicos y Coordenadas de Jacobi en Mecánica Celeste
19 de marzo 2026


## Sistemas Jerárquicos
En mecánica celeste, un sistema jerárquico es aquel donde las masas están organizadas en una estructura de escalas bien separadas. El ejemplo clásico es el sistema Sol-Tierra-Luna, donde:

La distancia Tierra-Luna (384,400 km) es mucho menor que la distancia Sol-Tierra (150 millones de km)
El período orbital Luna-Tierra (27.3 días) es mucho menor que el período Tierra-Sol (365.25 días)
Esta separación de escalas permite tratar el movimiento como una jerarquía de sistemas de dos cuerpos acoplados, simplificando enormemente el análisis del problema de N-cuerpos.

## Coordenadas de Jacobi
Las coordenadas de Jacobi son un sistema de coordenadas especialmente diseñado para sistemas jerárquicos. En lugar de usar coordenadas cartesianas individuales para cada cuerpo, se definen:

### Coordenadas relativas: Vectores que conectan pares de cuerpos en cada nivel jerárquico
Coordenadas del centro de masa: Posición del centro de masa de subsistemas
Ventajas principales:

### Separación natural: Los movimientos en diferentes escalas temporales se desacoplan
Conservación del momento: El momento total se conserva automáticamente
Estabilidad numérica: Reduce la acumulación de errores en integraciones de largo plazo
Reducción de dimensionalidad: Elimina grados de libertad asociados al movimiento del centro de masa total
Para el sistema Sol-Tierra-Luna, las coordenadas de Jacobi típicas serían:

r₁: Vector Tierra-Luna

r₂: Vector del centro de masa Tierra-Luna hacia el Sol

R: Posición del centro de masa total (que puede fijarse en el origen)


In [2]:
import pymcel as pc

In [4]:
import rebound as rb
import numpy as np
import matplotlib.pyplot as plt

Indicamos las coordenadas del primer subsistema en coordenadas del sistema, pero sólo los relativos entre sí, no respecto al CM aún.

Es importante definir primero la posición del centro de masa CENTRAL, mas no de los subsitstemas.

In [ ]:
G =1 # Manejaremos unidades canónicas 
m_Aa, m_Ab = 1, 0.01 # masas

# Comenzamos por el CM de todo el sistema, que es el punto de referencia para las coordenadas de Jacobi 
RCM_Aa = np.array([0, 0, 0])
VCM_Ab= np.array([0, 0, 0])

rCM_Ab = np.array([0.1, 0, 0])
vCM_Ab = np.array([0, 1, 0])

In [8]:
m_A = m_Aa + m_Ab # masa del primer subsistema
m_B = 1.0 # masa del segundo subsistema (un cuerpo externo)

# Describamos en Jacobi la ubicación del centro de masa de todo el sistema y su velocidad
RCM_AB = np.array([0, 0, 0])
VCM_AB = np.array([0, 0, 0])

# posición y velocidad relativa del subsistema A respecto al subsistema B
r_AB = np.array([1, 0, 0])
v_AB = np.array([0, 1, 0])

# Descendemos en el árbol jerárquico

r_B = RCM_AB - (m_A / (m_A + m_B)) * r_AB
v_B = VCM_AB - (m_A / (m_A + m_B)) * v_AB

RCM_A = RCM_AB + (m_B / (m_B + m_Ab)) * r_AB
VCM_A = VCM_AB + (m_B / (m_B + m_Ab)) * v_AB

In [9]:
# En el siguiente nivel, describimos el subsistema A en coordenadas de Jacobi respecto a su propio centro de masa
r_Aa = RCM_A + (m_Ab / (m_Aa + m_Ab)) * rCM_Ab
v_Aa = VCM_A + (m_Ab / (m_Aa + m_Ab)) * vCM_Ab

r_Ab = RCM_A - (m_Aa / (m_Aa + m_Ab)) * rCM_Ab # Note la diferencia de signo, porque el vector rCM_Ab va de Aa a Ab
v_Ab = VCM_A - (m_Aa / (m_Aa + m_Ab)) * vCM_Ab 

Hasta ahora, tenemos las posiciones y velocidades iniciales de los cuerpos. Ahora, podemos crear la simulación y el gráfico

In [10]:
# Create a rebound simulation with the three bodies
sim = rb.Simulation()
sim.G = G

# Add the three bodies using the computed positions and velocities
sim.add(m=m_Aa, x=r_Aa[0], y=r_Aa[1], z=r_Aa[2], vx=v_Aa[0], vy=v_Aa[1], vz=v_Aa[2])
sim.add(m=m_Ab, x=r_Ab[0], y=r_Ab[1], z=r_Ab[2], vx=v_Ab[0], vy=v_Ab[1], vz=v_Ab[2])
sim.add(m=m_B, x=r_B[0], y=r_B[1], z=r_B[2], vx=v_B[0], vy=v_B[1], vz=v_B[2])

# Integrate the system
times = np.linspace(0, 10, 500)
positions = np.zeros((3, len(times), 3))

for i, t in enumerate(times):
    sim.integrate(t)
    for j in range(3):
        positions[j, i] = [sim.particles[j].x, sim.particles[j].y, sim.particles[j].z]

# Plot with plotly for 3D visualization
import plotly.graph_objects as go

fig = go.Figure()

labels = ['Body Aa', 'Body Ab', 'Body B']
colors = ['blue', 'red', 'green']

for i, (label, color) in enumerate(zip(labels, colors)):
    fig.add_trace(go.Scatter3d(
        x=positions[i, :, 0],
        y=positions[i, :, 1],
        z=positions[i, :, 2],
        mode='lines',
        name=label,
        line=dict(color=color, width=4)
    ))

fig.update_layout(
    title='Hierarchical System Orbits (Jacobi Coordinates)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
    ),
    width=900,
    height=700
)

fig.show()